# Ingestion help: reFuel.ch

This script is an example how to convert a existing dataset into a 'unmapped_entity' using python script.

input:
 - reFuel.ch Tech data: Excel file

output:
 - a yaml file with 'unmapped_entity'

process:
 - read each row of the tech data and convert them in to unmapped data

TODO:
 - [2026-06-24]
    - rerun this notebook before go home due to a attribute was missing!
    - also run StorTech and EmbeddedCarbon
    - does the info in 'Metadata', 'Nomenclature', and 'Carrier' used? [revised and to be test; could increaes processing time]

In [1]:
## --- load packages
from pathlib import Path
import pandas as pd

import yaml

## Load target data (reFuel.ch TechDB excel)

In [ ]:
path_refuel = Path(r"raw_data\reFuel_TechDatabase_Clean_2026-06-03.xlsx")

refuel_data = pd.read_excel(path_refuel, sheet_name=None)

# manual fix: remove 0 from min_installation_size column in ConvTech sheet
col_num = refuel_data['ConvTech'].iloc[1].tolist().index('min_installation_size')
col_nam = refuel_data['ConvTech'].columns[col_num]

refuel_data['ConvTech'][col_nam] = refuel_data['ConvTech'][col_nam].replace(0, None)

# manual fix: remove 0 from operating_temperature_c column in ConvTech sheet
col_num = refuel_data['ConvTech'].iloc[1].tolist().index('operating_temperature_c')
col_nam = refuel_data['ConvTech'].columns[col_num]

refuel_data['ConvTech'][col_nam] = refuel_data['ConvTech'][col_nam].replace(0, None)

print(f"sheets found: {list(refuel_data.keys())}")

sheets found: ['Metadata', 'Nomenclature', 'Carrier', 'ConvTech', 'StorTech', 'EmbeddedCarbon', 'Reference']


In [3]:
refuel_data['ConvTech']

,Classification,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 30,Unnamed: 31,Unnamed: 32,Unnamed: 33,Unnamed: 34,Unnamed: 35,Unnamed: 36,Unnamed: 37,Unnamed: 38,Unnamed: 39
0,Technology ID,Technology Year,Technology Class,Description,Unit Operation,Tech Type,Reference Unit Size,Cost Base,Currency,Technology Readiness Level,...,OPEX per Energy,Discount Rate,Value Range Check,Uncertainty Rating,Mass Inflows,Mass Outflows,Energy Balance Error,Mass Balance Error,Balance Pass Flag,Source References
1,tech_id,tech_year,technology_class,description,unit_operation,tech_type,reference_unit_size,cost_base,currency,trl,...,opex_per_energy,discount_rate_pct,value_range_check,uncertainty_rating,mass_inflows,mass_outflows,energy_balance_error_pct,mass_balance_error_pct,balance_pass_flag,list_of_source_id
2,Electricity generation,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NH3_CCGT_El,2050,Gas turbine,NaN,CCGT,Conversion,NaN,ECA,EUR,9,...,0.002,NaN,true,NaN,"1216, 1714",1000,0.6906,0.0122,true,"""capex_per_capacity"", ""opex_per_capacity_yr"", ..."
4,NH3_OCGT_El,2050,Gas turbine,NaN,OCGT,Conversion,NaN,ECA,EUR,9,...,0.002,NaN,true,NaN,"1216, 1714",1000,0.6906,0.0122,true,"""capex_per_capacity"", ""opex_per_capacity_yr"", ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100,Air_DAC_CO2,2050,Carbon capture,Direct air capture using adsorption technology...,DAC (sorbent-based),Conversion,NaN,ECA,EUR,9,...,0,NaN,true,High,na,na,na,na,na,"""capex_per_capacity"", ""opex_per_capacity_yr"", ..."
101,CO2Flue_PSA_CO2,2030,Carbon capture,CO2 post combustion capture via Hot-Potassium ...,Pressure swing adsorption,Conversion,20,ECA,EUR,8,...,0.0022 EUR/kg,NaN,true,High,na,na,na,na,na,"""capex_per_capacity"", ""opex_per_capacity_yr"", ..."
102,CO2Flue_PSA_CO2,2040,Carbon capture,CO2 post combustion capture via Hot-Potassium ...,Pressure swing adsorption,Conversion,20,ECA,EUR,9,...,0.0022 EUR/kg,NaN,true,High,na,na,na,na,na,"""capex_per_capacity"", ""opex_per_capacity_yr"", ..."
103,CO2Flue_PSA_CO2,2050,Carbon capture,CO2 post combustion capture via Hot-Potassium ...,Pressure swing adsorption,Conversion,20,ECA,EUR,9,...,0.0022 EUR/kg,NaN,true,High,na,na,na,na,na,"""capex_per_capacity"", ""opex_per_capacity_yr"", ..."


In [4]:
## load reference data
df_ref = refuel_data['Reference'].copy()

df_ref.columns = df_ref.iloc[0]
df_ref = df_ref.iloc[1:].reset_index(drop=True)

df_ref

,source_id,description,reference_type,doi_or_url,comments,access_date
0,Allgoewer_2024,Paper by ETH from PATHFNDR,DOI,https://doi.org/10.1021/acs.iecr.4c01287,NaN,NaN
1,Rheintal_2025,"Rheintal paper of Empa, submitted 2025",DOI,https://dx.doi.org/10.2139/ssrn.5291581,NaN,NaN
2,biochar_jcp_2025,Financial feasibility of biochar production: A...,DOI,https://doi.org/10.1016/j.jclepro.2025.147167,NaN,NaN
3,book_hydropower_2020,Future Energy (Third Edition),DOI,https://doi.org/10.1016/B978-0-08-102886-5.000...,NaN,NaN
4,paper_CCGT+CCS_2016,Off-design point modelling of a 420 MW CCGT po...,DOI,https://doi.org/10.1016/j.apenergy.2016.06.087,This wasn't used anywhere in the db?,NaN
5,paper_CCGT_2024,Modeling combined-cycle power plants in a deta...,DOI,https://doi.org/10.1016/j.energy.2024.131246,NaN,NaN
6,paper_Msw to syngas,Municipal Solid Waste Gasification: Technologi...,DOI,https://doi.org/10.3390/su17156704,NaN,NaN
7,paper_PVcostProjection_2025,"review the cost projection of PV, wind, battery",DOI,https://doi.org/10.1016/j.apenergy.2025.125856,NaN,NaN
8,paper_RWGS,Technology Readiness and Emerging Prospects of...,DOI,https://doi.org/10.1002/cssc.202400865,NaN,NaN
9,paper_alpinePV_2024,ETHZ paper on Alpine PV cost and financial via...,DOI,https://doi.org/10.1016/j.apenergy.2024.124019,NaN,NaN


In [ ]:
## load attribute data

df_attr = refuel_data['Metadata'].copy()

# list of pre-selected attributes to be used for mapping
attrs = [
    'technical_efficiency', 'trl', 'tech_maturity', 'reference_unit_size', 'theoretical_efficiency',
    'operating_temperature_c', 'min_installation_size', 'lifetime_yr', 'capex_per_capacity', 'capex_one_time',
    'opex_fix_pct_of_capex', 'opex_per_capacity_yr', 'opex_per_energy', 'discount_rate_pct', 'uncertainty_rating',
    'storage_carrier', 'min_installation', 'charging_capacity_factor', 'discharging_capacity_factor', 
    'charging_efficiency', 'discharging_efficiency', 'min_soc', 'max_soc', 'standby_loss_per_hour',
    'capex_per_stor_capacity', 'opex_one_time', 'opex_per_stor_capacity_yr'
    ]

df_attr = df_attr[df_attr['Variable Name'].isin(attrs)].reset_index(drop=True)
df_attr.set_index('Variable Name', inplace=True)

df_attr = df_attr[['Column Header', 'Unit / Format', 'Allowed Values', 
                   'Description', 'Note']].copy()

df_attr

,Column Header,Unit / Format,Allowed Values,Description,Note
Variable Name,,,,,
reference_unit_size,Reference Unit Size,[MW] or [t/h],> 0,Nameplate capacity on which all specific cost ...,Capacity basis for scaling
trl,TRL,â€”,1â€“9,Technology Readiness Level on the standard 1â€“9...,See Nomenclature 4.2
tech_maturity,Technology Maturity,â€”,"{Prototype, Emerging, Mature}",Aggregated maturity classification derived fro...,See Nomenclature 4.2
technical_efficiency,Technical Efficiency,[0â€“1],0 < Î· â‰¤ 1,Overall energy conversion efficiency: main out...,"Based on kWh (electricity, heat) or LHV (chemi..."
theoretical_efficiency,Theoretical Efficiency,[0â€“1],0 < Î· â‰¤ 1,Ratio of the main output's energy content (LHV...,May be blank if not applicable
operating_temperature_c,Operating Temperature,[Â°C],â€”,Characteristic operating temperature of the co...,Relevant for thermal integration
min_installation_size,Minimum Installation Size,[kW] or [kg/h],â‰¥ 0,Smallest assumed unit capacity for the technol...,0 if no minimum constraint
storage_carrier,Storage Carrier,â€”,Carrier abbreviation,The energy or material carrier being stored. R...,NaN
min_installation,Minimum Installation,[kWh],â‰¥ 0,Smallest assumed storage capacity.,NaN


## Convert reFuel.ch tech to unmapped_entity

In [6]:
from ingestion_helper import *

In [7]:
sheet_name = 'ConvTech'

df_tech = prepare_df(refuel_data[sheet_name])

In [8]:
def refuel2unmapped(row):
    """Map a row from the ConvTech sheet to the unmapped format."""
    ue = {}     #unmapped entity

    ## 1. tech_name
    ue['technology_name'] = row.get("tech_id", "")

    ## 2. technology (specification)
    ue['technology'] = {
        "technology_description": clean(row.get("description")),
        "technology_type": clean(row.get("tech_type")),
        "technology_category": clean(row.get("technology_class")),
        "technology_assumption": None,
        "process_name": clean(row.get("unit_operation")),
        "process_type": None,
        "process_category": clean(row.get("tech_category")),
        "process_assumption": None,
        }        
    
    ## 3. scope
    ue["scope"] = {
        "geographic_scope_description":
            clean(row.get("cost_base")),    #!
        "temporal_scope_description":   #!
            str(row.get("tech_year"))
            if not is_nan(row.get("tech_year"))
            else None,
        "capacity_scope_description": clean(row.get("min_installation_size")),
        "system_boundary_description": clean(row.get("tech_boundary")),
        "scope_assumption": clean(row.get("tech_maturity"))
        }
    
    ## 4. sources
    ue = add_sources_to_record(ue, row.get("list_of_source_id"), df_ref)

    ## 5. attributes
    ue = add_attributes_to_record(ue, row, df_attr)

    ## 6. balancing
    ue = add_balancing_to_record(ue, row)

    ## 7. metadata
    ue["metadata"] = {
        "related_project": "reFuel.ch",
        "tags": ['Switzerland', 'power-to-X'],
        "other_notes": ['This unmapped entity was generated from the reFuel TechDatabase (2026-06-03) using the Motel ingestion pipeline.'],
    }

    return ue

In [9]:
## load the data from ConvTech sheet for testing

entities = []
sheet_name = 'ConvTech'

df_tech = prepare_df(refuel_data[sheet_name])

for indx in df_tech.index:
    row = df_tech.loc[indx]
    ue = refuel2unmapped(row)
    entities.append(ue)

entities

Cannot find source_id "linear_interpolation_2030_2050" in source dataset
Cannot find source_id "paper_biomass to syngas" in source dataset
Cannot find source_id "report_wood to hydrogen" in source dataset
Cannot find source_id "paper_biomass to syngas" in source dataset
Cannot find source_id "EP2050+Exkurs" in source dataset


[{'technology_name': 'NH3_CCGT_El',
  'technology': {'technology_description': None,
   'technology_type': 'Conversion',
   'technology_category': 'Gas turbine',
   'technology_assumption': None,
   'process_name': 'CCGT',
   'process_type': None,
   'process_category': None,
   'process_assumption': None},
  'scope': {'geographic_scope_description': 'ECA',
   'temporal_scope_description': '2050',
   'capacity_scope_description': None,
   'system_boundary_description': 'Plant ready to operate',
   'scope_assumption': 'Mature'},
  'sources': [{'source_name': 'report_power_to_ammonia_2018',
    'source_description': 'Power-to-ammonia in future North European 100 %  renewable power and heat system',
    'linked_attribute': ['capex_per_capacity',
     'lifetime_yr',
     'opex_per_capacity_yr',
     'opex_per_energy',
     'technical_efficiency']},
   {'source_name': 'conversion_parametrization',
    'source_description': 'Calculating Input Output Ratios of Reactions',
    'linked_attribut

In [10]:
## export entities in a yaml file
output_path = "unmapped_entities_refuel.yaml"

with open(output_path, "w", encoding="utf-8") as f:
    yaml.dump(entities, f, allow_unicode=True, sort_keys=False, default_flow_style=False)

print(f"Exported {len(entities)} entities to {output_path}")

Exported 99 entities to unmapped_entities_refuel.yaml


## Process StorTech sheet
Same structure as ConvTech; reuses `refuel2unmapped` directly. Balancing is empty (StorTech has no carrier flows).

In [ ]:
sheet_name = 'StorTech'
df_tech_stor = prepare_df(refuel_data[sheet_name])

stor_entities = []
for indx in df_tech_stor.index:
    row = df_tech_stor.loc[indx]
    ue = refuel2unmapped(row)
    stor_entities.append(ue)

print(f'StorTech entities: {len(stor_entities)}')
stor_entities

In [ ]:
output_path_stor = 'unmapped_entities_refuel_stortech.yaml'

with open(output_path_stor, 'w', encoding='utf-8') as f:
    yaml.dump(stor_entities, f, allow_unicode=True, sort_keys=False, default_flow_style=False)

print(f'Exported {len(stor_entities)} StorTech entities to {output_path_stor}')

## Process EmbeddedCarbon sheet
Different format: each row is one technology with 8 LCA values (2 scenarios x 4 years).
Mapped as one unmapped entity per row; each scenario-year value becomes a time-indexed attribute.
- attribute_name: embedded_carbon_ssp2_ndc or embedded_carbon_ssp2_pkbudg1000
- time_index: 2025 / 2030 / 2040 / 2050
- notes: unit, ref_product, lca_location, scenario label

In [ ]:
_SCENARIOS = {
    'ssp2_ndc':        ['ssp2_ndc_2025',        'ssp2_ndc_2030',        'ssp2_ndc_2040',        'ssp2_ndc_2050'],
    'ssp2_pkbudg1000': ['ssp2_pkbudg1000_2025',  'ssp2_pkbudg1000_2030',  'ssp2_pkbudg1000_2040',  'ssp2_pkbudg1000_2050'],
}
_YEARS = [2025, 2030, 2040, 2050]

def embeddedcarbon2unmapped(row):
    ue = {}
    ue['technology_name'] = row.get('tech_id', '')

    ue['technology'] = {
        'technology_description': clean(row.get('lca_activity')),
        'technology_type':        clean(row.get('tech_type')),
        'technology_category':    None,
        'technology_assumption':  None,
        'process_name':           clean(row.get('ref_product')),
        'process_type':           None,
        'process_category':       None,
        'process_assumption':     None,
    }

    ue['scope'] = {
        'geographic_scope_description': clean(row.get('lca_location')),
        'temporal_scope_description':   None,
        'capacity_scope_description':   None,
        'system_boundary_description':  None,
        'scope_assumption':             None,
    }

    ue['sources'] = []

    lca_unit = clean(row.get('lca_unit'))
    notes_base = (
        f'lca_unit: {lca_unit}'
        f' | ref_product: {clean(row.get("ref_product"))}'
        f' | lca_location: {clean(row.get("lca_location"))}'
    )
    if not is_nan(row.get('notes')):
        notes_base += f' | notes: {row.get("notes")}'

    attributes = []
    for scenario_key, cols in _SCENARIOS.items():
        for year, col in zip(_YEARS, cols):
            val = clean(row.get(col))
            if val is not None:
                attributes.append({
                    'attribute_name':    'embedded_carbon',
                    'value':             val,
                    'uncertainty_notes': f'climate scenario: {scenario_key}',
                    'time_index':        year,
                    'notes':             notes_base,
                })
    ue['attributes'] = attributes

    ue['balancing'] = {'inputs': [], 'outputs': []}

    ue['metadata'] = {
        'related_project': 'reFuel.ch',
        'tags': ['Switzerland', 'power-to-X', 'embedded-carbon', 'LCA'],
        'other_notes': ['Generated from reFuel TechDatabase (2026-06-03) EmbeddedCarbon sheet.'],
    }

    return ue


In [ ]:
df_ec = prepare_df(refuel_data['EmbeddedCarbon'])

ec_entities = []
for indx in df_ec.index:
    row = df_ec.loc[indx]
    ue = embeddedcarbon2unmapped(row)
    ec_entities.append(ue)

print(f'EmbeddedCarbon entities: {len(ec_entities)}')
ec_entities

In [ ]:
output_path_ec = 'unmapped_entities_refuel_embeddedcarbon.yaml'

with open(output_path_ec, 'w', encoding='utf-8') as f:
    yaml.dump(ec_entities, f, allow_unicode=True, sort_keys=False, default_flow_style=False)

print(f'Exported {len(ec_entities)} EmbeddedCarbon entities to {output_path_ec}')